# 10 Minutes to Dask-cuDF

This is a short introduction to Dask-cuDF for distributed GPU DataFrames. For single-GPU examples, see [10 Minutes to cuDF](10min.ipynb).

[Dask-cuDF](https://docs.nvidia.com/dask-cudf/latest/) uses cuDF DataFrames as partitions of a Dask DataFrame. It can distribute a workflow across multiple GPUs and process datasets larger than a single GPU's memory. Like Dask, it builds task graphs lazily and executes them when results are requested.

The examples below create small cuDF objects to explain the API. For larger datasets, read directly with Dask-cuDF's I/O functions, as shown later in this notebook.

This notebook uses the explicit `dask_cudf` API for clarity. For applications, we recommend setting Dask's `dataframe.backend` configuration option to `"cudf"` and using the `dask.dataframe` API directly. See the [Dask-cuDF documentation](https://docs.nvidia.com/dask-cudf/latest/) for setup instructions.


In [1]:
import os

import cudf
import cupy as cp
import dask.dataframe as dd
import dask_cudf
import pandas as pd

cp.random.seed(12)

#### Portions of this were borrowed and adapted from the
#### cuDF cheatsheet, existing cuDF documentation,
#### and 10 Minutes to Pandas.

## Object Creation

Creating a `cudf.Series` and `dask_cudf.Series`.

In [2]:
s = cudf.Series([1, 2, 3, None, 4])
s

0       1
1       2
2       3
3    <NA>
4       4
dtype: int64

In [3]:
ds = dask_cudf.from_cudf(s, npartitions=2)
# Note the call to head here to show the first few entries, unlike
# cuDF objects, Dask-cuDF objects do not have a printing
# representation that shows values since they may not be in local
# memory.
ds.head(n=3)

0    1
1    2
2    3
dtype: int64

Creating a `cudf.DataFrame` and a `dask_cudf.DataFrame` by specifying values for each column.

In [4]:
df = cudf.DataFrame(
    {
        "a": list(range(20)),
        "b": list(reversed(range(20))),
        "c": list(range(20)),
    }
)
df

,a,b,c
0,0,19,0
1,1,18,1
2,2,17,2
3,3,16,3
4,4,15,4
5,5,14,5
6,6,13,6
7,7,12,7
8,8,11,8
9,9,10,9


Now we will convert our cuDF dataframe into a Dask-cuDF equivalent. Here we call out a key difference: to inspect the data we must call a method (here `.head()` to look at the first few values). In the general case (see the end of this notebook), the data in `ddf` will be distributed across multiple GPUs.

In this small case, we could call `ddf.compute()` to obtain a cuDF object from the Dask-cuDF object. In general, we should avoid calling `.compute()` on large dataframes, and restrict ourselves to using it when we have some (relatively) small postprocessed result that we wish to inspect. Hence, throughout this notebook we will generally call `.head()` to inspect the first few values of a Dask-cuDF dataframe, occasionally calling out places where we use `.compute()` and why.

*To understand more of the differences between how cuDF and Dask cuDF behave here, visit the [10 Minutes to Dask](https://docs.dask.org/en/stable/10-minutes-to-dask.html) tutorial after this one.*

In [5]:
ddf = dask_cudf.from_cudf(df, npartitions=2)
ddf.head()

,a,b,c
0,0,19,0
1,1,18,1
2,2,17,2
3,3,16,3
4,4,15,4


Creating a `cudf.DataFrame` from a pandas `Dataframe` and a `dask_cudf.Dataframe` from a `cudf.Dataframe`.

*Note that best practice for using dask-cuDF is to read data directly into a `dask_cudf.DataFrame` with `read_csv` or other builtin I/O routines (discussed below).*

In [6]:
pdf = pd.DataFrame({"a": [0, 1, 2, 3], "b": [0.1, 0.2, None, 0.3]})
gdf = cudf.DataFrame(pdf)
gdf

,a,b
0,0,0.1
1,1,0.2
2,2,NaN
3,3,0.3


In [7]:
dask_gdf = dask_cudf.from_cudf(gdf, npartitions=2)
dask_gdf.head(n=2)

,a,b
0,0,0.1
1,1,0.2


## Viewing Data

Viewing the top rows of a GPU dataframe.

In [8]:
ddf.head(2)

,a,b,c
0,0,19,0
1,1,18,1


Sorting by values.

In [9]:
ddf.sort_values(by="b").head()

,a,b,c
19,19,0,19
18,18,1,18
17,17,2,17
16,16,3,16
15,15,4,15


## Selecting a Column

Selecting a single column, which initially yields a `cudf.Series` or `dask_cudf.Series`. Calling `compute` results in a `cudf.Series` (equivalent to `df.a`).

In [10]:
ddf["a"].head()

0    0
1    1
2    2
3    3
4    4
Name: a, dtype: int64

## Selecting Rows by Label

Selecting rows from index 2 to index 5 from columns 'a' and 'b'.

In [11]:
ddf.loc[2:5, ["a", "b"]].head()

,a,b
2,2,17
3,3,16
4,4,15
5,5,14


## Selecting Rows by Position

Selecting via integers and integer slices, like numpy/pandas. Note that this functionality is not available for Dask-cuDF DataFrames.

## Boolean Indexing

Selecting rows in a `DataFrame` or `Series` by direct Boolean indexing.

In [12]:
ddf[ddf.b > 15].head(n=3)

,a,b,c
0,0,19,0
1,1,18,1
2,2,17,2


Selecting values from a `DataFrame` where a Boolean condition is met, via the `query` API.

Note here we call `compute()` rather than `head()` on the Dask-cuDF dataframe since we are happy that the number of matching rows will be small (and hence it is reasonable to bring the entire result back).

In [13]:
ddf.query("b == 3").compute()

/home/coder/.conda/envs/rapids/lib/python3.14/site-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


,a,b,c
16,16,3,16


You can also pass local variables to Dask-cuDF queries, via the `local_dict` keyword. With standard cuDF, you may either use the `local_dict` keyword or directly pass the variable via the `@` keyword. Supported logical operators include `>`, `<`, `>=`, `<=`, `==`, and `!=`.

In [14]:
dask_cudf_comparator = 3
ddf.query("b == @val", local_dict={"val": dask_cudf_comparator}).compute()

,a,b,c
16,16,3,16


Missing Data
------------

Missing data can be replaced by using the `fillna` method.

In [15]:
ds.fillna(999).head(n=4, npartitions=-1)

0      1
1      2
2      3
3    999
dtype: int64

## Stats

Calculating descriptive statistics for a `Series`.

This serves as a prototypical example of when we might want to call `.compute()`. The result of computing the mean and variance is a single number in each case, so it is definitely reasonable to look at the entire result!

In [16]:
ds.mean().compute(), ds.var().compute()

(np.float64(2.5), np.float64(1.6666666666666667))

## Applying Functions

Applying functions to a `Series`. Note that applying user defined functions directly with Dask cuDF is not yet implemented. For now, you can use [map_partitions](https://docs.dask.org/en/stable/generated/dask.dataframe.DataFrame.map_partitions.html) to apply a function to each partition of the distributed dataframe.

In [17]:
def add_ten(num):
    return num + 10

In [18]:
ddf["a"].map_partitions(add_ten).head()

0    10
1    11
2    12
3    13
4    14
Name: a, dtype: int64

## Histogramming

Counting the number of occurrences of each unique value of variable.

In [19]:
ddf.a.value_counts().head()

a
0     1
2     1
4     1
9     1
13    1
Name: count, dtype: int64

## String Methods

Like pandas, cuDF provides string processing methods in the `str` attribute of `Series`. Full documentation of string methods is a work in progress. Please see the [cuDF API documentation](https://docs.nvidia.com/cudf/latest/cudf/api_docs/series/#string-handling) for more information.

In [20]:
s = cudf.Series(["A", "B", "C", "Aaba", "Baca", None, "CABA", "dog", "cat"])

In [21]:
ds = dask_cudf.from_cudf(s, npartitions=2)
ds.str.lower().head(n=4)

0       a
1       b
2       c
3    aaba
dtype: string

As well as simple manipulation, We can also match strings using [regular expressions](https://docs.nvidia.com/cudf/latest/cudf/api_docs/api/cudf.core.accessors.string.StringMethods.match/).

In [22]:
ds.str.match("^[aAc].+").head()

0    False
1    False
2    False
3     True
4    False
dtype: bool[pyarrow]

## Concat

Concatenating `Series` and `DataFrames` row-wise.

In [23]:
s = cudf.Series([1, 2, 3, None, 5])

In [24]:
ds2 = dask_cudf.from_cudf(s, npartitions=2)
dask_cudf.concat([ds2, ds2]).compute()

0       1
1       2
2       3
3    <NA>
4       5
0       1
1       2
2       3
3    <NA>
4       5
dtype: int64

## Join

Performing SQL style merges. Note that the dataframe order is **not maintained**, but may be restored post-merge by sorting by the index.

In [25]:
df_a = cudf.DataFrame()
df_a["key"] = ["a", "b", "c", "d", "e"]
df_a["vals_a"] = [float(i + 10) for i in range(5)]

df_b = cudf.DataFrame()
df_b["key"] = ["a", "c", "e"]
df_b["vals_b"] = [float(i + 100) for i in range(3)]

In [26]:
ddf_a = dask_cudf.from_cudf(df_a, npartitions=2)
ddf_b = dask_cudf.from_cudf(df_b, npartitions=2)

merged = ddf_a.merge(ddf_b, on=["key"], how="left").head(n=4, npartitions=-1)
merged

,key,vals_a,vals_b
0,a,10.0,100.0
0,d,13.0,NaN
1,e,14.0,102.0
2,b,11.0,NaN


## Grouping

Like [pandas](https://pandas.pydata.org/docs/user_guide/groupby.html), cuDF and Dask-cuDF support the [Split-Apply-Combine groupby paradigm](https://doi.org/10.18637/jss.v040.i01).

In [27]:
df["agg_col1"] = [1 if x % 2 == 0 else 0 for x in range(len(df))]
df["agg_col2"] = [1 if x % 3 == 0 else 0 for x in range(len(df))]

ddf = dask_cudf.from_cudf(df, npartitions=2)

Grouping and then applying the `sum` function to the grouped data.

In [28]:
ddf.groupby("agg_col1").sum().compute()

,a,b,c,agg_col2
agg_col1,,,,
1,90,100,90,4
0,100,90,100,3


Grouping hierarchically then applying the `sum` function to grouped data.

In [29]:
ddf.groupby(["agg_col1", "agg_col2"]).sum().compute()

,,a,b,c
agg_col1,agg_col2,,,
1,1,36,40,36
0,0,73,60,73
1,0,54,60,54
0,1,27,30,27


Grouping and applying statistical functions to specific columns, using `agg`.

In [30]:
ddf.groupby("agg_col1").agg({"a": "max", "b": "mean", "c": "sum"}).compute()

,a,b,c
agg_col1,,,
1,18,10.0,90
0,19,9.0,100


## Time Series

`DataFrames` supports `datetime` typed columns, which allow users to interact with and filter data based on specific timestamps.

In [31]:
import datetime as dt

date_df = cudf.DataFrame()
date_df["date"] = pd.date_range("11/20/2018", periods=72, freq="D")
date_df["value"] = cp.random.sample(len(date_df))

search_date = dt.datetime.strptime("2018-11-23", "%Y-%m-%d")

In [32]:
date_ddf = dask_cudf.from_cudf(date_df, npartitions=2)
date_ddf.query(
    "date <= @search_date", local_dict={"search_date": search_date}
).compute()

,date,value
0,2018-11-20,0.082654
1,2018-11-21,0.399417
2,2018-11-22,0.784297
3,2018-11-23,0.070303


## Categoricals

`DataFrames` support categorical columns.

In [33]:
gdf = cudf.DataFrame(
    {"id": [1, 2, 3, 4, 5, 6], "grade": ["a", "b", "b", "a", "a", "e"]}
)
gdf["grade"] = gdf["grade"].astype("category")
gdf

,id,grade
0,1,a
1,2,b
2,3,b
3,4,a
4,5,a
5,6,e


In [34]:
dgdf = dask_cudf.from_cudf(gdf, npartitions=2)
dgdf.head(n=3)

,id,grade
0,1,a
1,2,b
2,3,b


Accessing the underlying code values of each categorical observation.

In [35]:
dgdf.grade.cat.codes.compute()

0    0
1    1
2    1
3    0
4    0
5    2
dtype: int8

## Converting to Pandas

Converting a cuDF and Dask-cuDF `DataFrame` to a pandas `DataFrame`.

To convert the first few entries to pandas, we similarly call `.head()` on the Dask-cuDF dataframe to obtain a local cuDF dataframe, which we can then convert.

In [36]:
ddf.head().to_pandas()

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0


In contrast, if we want to convert the entire frame, we need to call `.compute()` on `ddf` to get a local cuDF dataframe, and then call `to_pandas()`, followed by subsequent processing. This workflow is less recommended, since it both puts high memory pressure on a single GPU (the `.compute()` call) and does not take advantage of GPU acceleration for processing (the computation happens on in pandas).

In [37]:
ddf.compute().to_pandas().head()

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0


## Converting to Numpy

Converting a cuDF or Dask-cuDF `DataFrame` to a numpy `ndarray`.

In [38]:
ddf.compute().to_numpy()

array([[ 0, 19,  0,  1,  1],
       [ 1, 18,  1,  0,  0],
       [ 2, 17,  2,  1,  0],
       [ 3, 16,  3,  0,  1],
       [ 4, 15,  4,  1,  0],
       [ 5, 14,  5,  0,  0],
       [ 6, 13,  6,  1,  1],
       [ 7, 12,  7,  0,  0],
       [ 8, 11,  8,  1,  0],
       [ 9, 10,  9,  0,  1],
       [10,  9, 10,  1,  0],
       [11,  8, 11,  0,  0],
       [12,  7, 12,  1,  1],
       [13,  6, 13,  0,  0],
       [14,  5, 14,  1,  0],
       [15,  4, 15,  0,  1],
       [16,  3, 16,  1,  0],
       [17,  2, 17,  0,  0],
       [18,  1, 18,  1,  1],
       [19,  0, 19,  0,  0]])

Converting a cuDF or Dask-cuDF `Series` to a numpy `ndarray`.

In [39]:
ddf["a"].compute().to_numpy()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19])

## Converting to Arrow

Converting a cuDF or Dask-cuDF `DataFrame` to a PyArrow `Table`.

In [40]:
ddf.head().to_arrow()

pyarrow.Table
a: int64
b: int64
c: int64
agg_col1: int64
agg_col2: int64
----
a: [[0,1,2,3,4]]
b: [[19,18,17,16,15]]
c: [[0,1,2,3,4]]
agg_col1: [[1,0,1,0,1]]
agg_col2: [[1,0,0,1,0]]

## Reading/Writing CSV Files

Writing to a CSV file.

In [41]:
os.makedirs("example_output", exist_ok=True)

In [42]:
ddf.compute().to_csv("example_output/foo_dask.csv", index=False)

Reading from a csv file.

Note that for the Dask-cuDF case, we use `dask_cudf.read_csv` in preference to `dask_cudf.from_cudf(cudf.read_csv)` since the former can parallelize across multiple GPUs and handle larger CSV files that would fit in memory on a single GPU.

In [43]:
ddf = dask_cudf.read_csv("example_output/foo_dask.csv")
ddf.head()

[2103541][00:12:00:799468][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0


Reading all CSV files in a directory into a single `dask_cudf.DataFrame`, using the star wildcard.

In [44]:
ddf = dask_cudf.read_csv("example_output/*.csv")
ddf.head()

[2103541][00:12:00:836279][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0


## Reading/Writing Parquet Files

Writing to parquet files from a `dask_cudf.DataFrame` using cuDF's parquet writer under the hood.

In [45]:
ddf.to_parquet("example_output/ddf_parquet_files")

## Dask Performance Tips

Like Apache Spark, Dask operations are [lazy](https://en.wikipedia.org/wiki/Lazy_evaluation). Instead of being executed immediately, most operations are added to a task graph and the actual evaluation is delayed until the result is needed.

Calling `persist` on a Dask collection starts computation and retains the resulting partitions on the workers. With a distributed client, it returns before computation finishes. Calling `wait` on the persisted collection blocks the caller until its futures complete. Dask's scheduler already respects dependencies between tasks; use `wait` when a subsequent client-side action, such as measuring GPU memory usage, needs the computation to have finished.

The snippets below provide basic examples, using `LocalCUDACluster` to create one dask-worker per GPU on the local machine. For more detailed information about `persist` and `wait`, please see the Dask documentation for [persist](https://docs.dask.org/en/latest/api.html#dask.persist) and [wait](https://docs.dask.org/en/latest/futures.html#distributed.wait). Wait relies on the concept of Futures, which is beyond the scope of this tutorial. For more information on Futures, see the Dask [Futures](https://docs.dask.org/en/latest/futures.html) documentation. For more information about multi-GPU clusters, please see the [dask-cuda](https://github.com/rapidsai/dask-cuda) library (documentation is in progress).

First, we set up a GPU cluster. With our `client` set up, Dask-cuDF computation will be distributed across the GPUs in the cluster.

In [46]:
import time

from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster

cluster = LocalCUDACluster()
client = Client(cluster)

### Persisting Data

Next, we create partitions directly on the workers rather than sending a large cuDF DataFrame from the client. The empty cuDF DataFrame passed as `meta` describes each partition's columns and types without computing the data. We then add a column lazily.

In [47]:
nrows = 10000000
npartitions = 16


def make_partition(partition):
    start = partition * (nrows // npartitions)
    values = cp.arange(start, start + nrows // npartitions)
    return cudf.DataFrame({"a": values, "b": values})


ddf2 = dd.from_map(
    make_partition,
    range(npartitions),
    meta=cudf.DataFrame({"a": [], "b": []}, dtype="int64"),
)
ddf2["c"] = ddf2["a"] + 5
ddf2

,a,b,c
npartitions=16,,,
,int64,int64,int64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [48]:
!nvidia-smi

Wed Nov 18 00:12:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           On  |   00000000:06:00.0 Off |                    0 |
| N/A   31C    P0             61W /  300W |     638MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

|   5  Tesla V100-SXM2-32GB           On  |   00000000:86:00.0 Off |                    0 |
| N/A   31C    P0             41W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   6  Tesla V100-SXM2-32GB           On  |   00000000:89:00.0 Off |                    0 |
| N/A   31C    P0             44W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   7  Tesla V100-SXM2-32GB           On  |   00000000:8A:00.0 Off |                    0 |
| N/A   29C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |            

Because Dask is lazy, the transformation has not yet been fully computed. Calling `persist` starts computing the collection and keeps the resulting partitions on the workers. The resulting task graph references those partitions instead of repeating the transformation.

In [49]:
ddf2 = ddf2.persist()
ddf2

,a,b,c
npartitions=16,,,
,int64,int64,int64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [50]:
_ = wait(ddf2)
!nvidia-smi

Wed Nov 18 00:12:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           On  |   00000000:06:00.0 Off |                    0 |
| N/A   31C    P0             61W /  300W |     702MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

|   2  Tesla V100-SXM2-32GB           On  |   00000000:0A:00.0 Off |                    0 |
| N/A   31C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   3  Tesla V100-SXM2-32GB           On  |   00000000:0B:00.0 Off |                    0 |
| N/A   29C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   4  Tesla V100-SXM2-32GB           On  |   00000000:85:00.0 Off |                    0 |
| N/A   31C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |            

Because we forced computation, we now have a larger object in distributed GPU memory. Note that actual numbers will differ between systems (for example depending on how many devices are available).

### Wait
Depending on our workflow or distributed computing setup, we may want to `wait` until all upstream tasks have finished before proceeding with a specific function. This section shows an example of this behavior, adapted from the Dask documentation.

First, we create a new Dask DataFrame and define a function that we'll map to every partition in the dataframe.

In [51]:
import random

ddf1 = ddf2[["a", "b"]].repartition(npartitions=10)


def func(df):
    time.sleep(random.uniform(0.1, 1))
    return (df + 5) * 3 - 11

This function transforms every column, with a short random delay to simulate partitions taking different amounts of time. We apply it to every partition using `map_partitions`, then call `persist` to start execution.

In [52]:
results_ddf = ddf1.map_partitions(func)
results_ddf = results_ddf.persist()

However, some partitions will be done **much** sooner than others. If we had downstream processes that should wait for all partitions to be completed, we can enforce that behavior using `wait`.

In [53]:
_ = wait(results_ddf)

With `wait` completed, we can safely proceed on in our workflow.

Close the client and cluster when they are no longer needed.

In [54]:
client.close()
cluster.close()